# Mississippi Basin

Welcome to the real-world application tutorial! In this guide, we walk through a complete, end-to-end data pipeline using the Mississippi River Basin. You will learn how to obtain external land surface model data, couple it to the river network, and run a full RAPID2 simulation.

This tutorial is provided as an interactive python notebook (`ipynb`), and can be opend and run in your browser by clicking the Binder badge:
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/c-h-david/rapid-hub/mississippi_NB_updates?labpath=docs/user-guide/examples/tutorial-mississippi.ipynb)

All commands used in this tutorial are written in `bash` and can also be run by copying them into your terminal. Note that the `%%bash` at the start of most code cells simply converts the cell from `python` to `bash`.

> ⚠️ Binder version still under developement and may not function properly.


Before starting, esnure that RAPID2 has been installed in your vistual environment (see the [Quickstart Guide](https://c-h-david.github.io/rapid-hub/user-guide/quick-start-sandbox/)). If running this tutorial in Binder, this setup is done automatically. We can check that rapid2 is installed and verify the version:

In [ ]:
%%bash
rapid2 --version

## 1. Get Static Parameter Files for the Mississippi River

First, let's create working directories where the inputs and outputs of RAPID2 will be saved.

In [ ]:
%%bash
mkdir -p input/Tutorial
mkdir -p output/Tutorial

Next, we need to get six static parameter files for the Mississippi Basin. We'll be using prepared files from a Zenodo repository:

[RAPID2 static files for global implementation with MERIT Basins v1.0 and GLDAS v2.0](https://zenodo.org/records/20672740)

This repository includes 61 zipped folders corresponding to the global Pfafstetter Level 2 hydrologic regions. For this tutorial, we'll use the `pfaf_74` files, corresponding to the Mississippi
River Basin. We can download `MERIT_Basins_v1.0_pfaf_74_GLDAS_v2.0_RAPID_v2.0.zip` using `wget`, and then extract the files using `unzip`.

In [ ]:
%%bash
wget -P input/Tutorial/ https://zenodo.org/records/20672740/files/MERIT_Basins_v1.0_pfaf_74_GLDAS_v2.0_RAPID_v2.0.zip
unzip input/Tutorial/MERIT_Basins_v1.0_pfaf_74_GLDAS_v2.0_RAPID_v2.0.zip -d input/Tutorial/

After opening the folder, it contains six `parquet` files needed to run RAPID2. Here is a quick overview of the input files:

1. `bas_pfaf_74_topo.parquet`: List of reaches within the basin.
2. `con_pfaf_74.parquet`: Network topology of the reaches, representing upstream/downstream connectivity.
3. `cpl_pfaf_74_GLDAS.parquet`: Spatial linkage between the land surface model outputs and hydrography network.
4. `crd_pfaf_74.parquet`: Coordinates of each reach polyline.
5. `kpr_pfaf_74_nrm.parquet`: Parameter *k* that represents the flow wave travel time.
6. `xpr_pfaf_74_nrm.parquet`: Dimensionless parameter *x* that relates to flow wave attenuation.

Files 1-4 correspond to the hydrography of the MERIT-Hydro v0.7 Basins v1.0. dataset. Meanwhile, files 5-6 include reach-specific values of *k* and *x*, which are Muskingum routing parameters.

> Note: It is possible to adjust the residence times related to the *k* and *x* parameters. Three different residence times are supported: low (`low`), medium (`nrm`), and high (`high`). These files can also be found [on Zenodo](https://doi.org/10.5281/zenodo.8248069) within zipped folders corresponding to the Pfafstetter Level 2 basins (e.g., `k_pfaf_ii_low.zip`, `x_pfaf_ii_hig.zip`). For more information, please see [Collins et al., 2024. *Nature Geoscience.*](https://doi.org/10.1038/s41561-024-01421-5)


## 2. Download Raw Runoff Data with `dgldas2`

The RAPID2 routing model requires runoff inputs from a land surface model. We'll use runoff data downloaded from the Global Land Data Assimilation System (GLDAS) using the `dgldas2` command that is part of the `rapid2` package.

A NASA Earthdata account is required to download the GLDAS data. If you have not yet done so, you may also need to accept an end-user license agreement from the NASA Goddard Earth Sciences (GES) Data and Information Services Center (DISC).

For this tutorial, we'll download GLDAS phase `2.1`, model `VIC`, for `2010-01`, using the following command.

> ⚠️ If running this tutorial in Binder, you first need to import the `earthaccess` python package and run the login routine. This will ask you for your Earthdata login and password, and then save your redentials to a .netrc file. If running everything in your terminal, the `dgldas` command will handle your login credentials and the earthaccess code can be skipped.

In [ ]:
# Skip this cell if running in terminal
import earthaccess
earthaccess.login(persist=True)

The follwing `dgldas2` command will download runoff from GLDAS phase `2.1`, using the `VIC` model, for January, 2010:

In [ ]:
%%bash
dgldas2 \
    --phase 2.1 \
    --model VIC \
    --time 2010-01 \
    --land_surface_model \
    input/Tutorial/GLDAS_2.1_VIC_2010-01.nc4

The new file is placed in `input/Tutorial`. Other files are also automatically
downloaded in that directory and removed after use.

> Note: GLDAS `2.0` is also supported, as are the `CLSM` and `NOAH` models.

## 3. Coupling the Data with `cpllsm`

Once you have the raw land surface model data and the static river basin parameters, the two must be linked by mapping the runoff to the river network.. The `cpllsm` tool transforms the gridded runoff data into RAPID2's reach-based external inflow format. By providing the land surface model data, connectivity, coordinates, and coupling files, this tool generates the external inflow NetCDF file required for routing.

In [ ]:
%%bash
cpllsm \
    --land_surface_model \
    input/Tutorial/GLDAS_2.1_VIC_2010-01.nc4 \
    --connectivity \
    input/Tutorial/con_pfaf_74.parquet \
    --coordinates \
    input/Tutorial/crd_pfaf_74.parquet \
    --coupling \
    input/Tutorial/cpl_pfaf_74_GLDAS.parquet \
    --external_inflow \
    input/Tutorial/Qext_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4

## 4. Create Cold Start File with `zeroqinit`

At the start of our simulation, we need to provide RAPID2 with initial values of discharge. We'll use the next command to create a cold start file with all zeroes.

In [ ]:
%%bash
zeroqinit \
    --external_inflow \
    input/Tutorial/Qext_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4 \
    --initial_outflow \
    input/Tutorial/Qinit_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4

## 5. Run the Routing Model with `rapid2`

RAPID2 uses a YAML configuration file (namelist) to specify input files, parameters, and output locations. `IS_dtR` is the routing time step (in seconds).

Before running the model, we need to generate the namelist file for our tutorial run. We can do this using the `cat` command to concatenate lines of text into a single .yml file. 

In [ ]:
%%bash
cat << 'EOF' > input/Tutorial/namelist_Tutorial.yml
# -----------------------------------------------------------------------------
# Mandatory input files
# -----------------------------------------------------------------------------
Qex_ncf: './input/Tutorial/Qext_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4'
Q00_ncf: './input/Tutorial/Qinit_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4'

con_pqt: './input/Tutorial/con_pfaf_74.parquet'
kpr_pqt: './input/Tutorial/kpr_pfaf_74_nrm.parquet'
xpr_pqt: './input/Tutorial/xpr_pfaf_74_nrm.parquet'

bas_pqt: './input/Tutorial/bas_pfaf_74_topo.parquet'

# -----------------------------------------------------------------------------
# Mandatory values
# -----------------------------------------------------------------------------
IS_dtR: 900

# -----------------------------------------------------------------------------
# Mandatory output files
# -----------------------------------------------------------------------------
Qou_ncf: './output/Tutorial/Qout_pfaf_74_GLDAS_2.1_VIC_2010-01_tst.nc4'
Qfi_ncf: './output/Tutorial/Qfinal_pfaf_74_GLDAS_2.1_VIC_2010-01_tst.nc4'
EOF

The output namelist file should look like this:
```yaml
# -----------------------------------------------------------------------------
# Mandatory input files
# -----------------------------------------------------------------------------
Qex_ncf: './input/Tutorial/Qext_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4'
Q00_ncf: './input/Tutorial/Qinit_pfaf_74_GLDAS_2.1_VIC_2010-01.nc4'

con_pqt: './input/Tutorial/con_pfaf_74.parquet'
kpr_pqt: './input/Tutorial/kpr_pfaf_74_nrm.parquet'
xpr_pqt: './input/Tutorial/xpr_pfaf_74_nrm.parquet'

bas_pqt: './input/Tutorial/bas_pfaf_74_topo.parquet'

# -----------------------------------------------------------------------------
# Mandatory values
# -----------------------------------------------------------------------------
IS_dtR: 900

# -----------------------------------------------------------------------------
# Mandatory output files
# -----------------------------------------------------------------------------
Qou_ncf: './output/Tutorial/Qout_pfaf_74_GLDAS_2.1_VIC_2010-01_tst.nc4'
Qfi_ncf: './output/Tutorial/Qfinal_pfaf_74_GLDAS_2.1_VIC_2010-01_tst.nc4'
```

Once the namelist file is created, it's time to run RAPID2! The basic command is:

In [ ]:
%%bash
rapid2 --namelist input/Tutorial/namelist_Tutorial.yml

That's it!

Our output files were written to the `output/Tutorial` folder. `Qout_pfaf_74_GLDAS_2.1_VIC_2010-01_tst.nc4` represents the discharge time series for the Mississippi River Basin in January 2010. Meanwhile, `Qfinal_pfaf_74_GLDAS_2.1_VIC_2010-01_tst.nc4` gives us the final state for a model restart.

> Note: To run RAPID2 for longer than one month, the simulation can be extended by using the Qfinal file from the previous time step (`Qfi_ncf`) as the Qinit file for the next time step (`Q00_ncf`). It is currently possible in RAPID2 to download up to 300 time steps of runoff data from GLDAS at once, which is about 37.5 days of data.